# Phase 1 — Data Cleaning Template

## Objectif
Nettoyer un dataset brut pour le rendre fiable, exploitable et documenté.

## Pipeline
1. Charger les données
2. Comprendre la structure
3. Faire un profiling
4. Définir les problèmes
5. Appliquer les corrections
6. Valider le résultat
7. Exporter les données nettoyées
8. Documenter les transformations

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
import re

In [8]:
# -----------------------------
# PARAMÈTRES À ADAPTER
# -----------------------------

RAW_PATH = "dirty_cafe_sales.csv"         # <-- adapte le chemin
CLEAN_PATH = "cafe_dataset_cleaned.csv"

# Colonnes à adapter selon ton dataset
ID_COLUMNS = []            # ex: ["customer_id"]
DATE_COLUMNS = []          # ex: ["order_date", "delivery_date"]
CATEGORICAL_COLUMNS = []   # ex: ["city", "gender", "status"]
NUMERIC_COLUMNS = []       # ex: ["age", "price", "amount"]
TEXT_COLUMNS = []          # ex: ["comments", "description"]

# Colonnes obligatoires
REQUIRED_COLUMNS = []      # ex: ["customer_id", "order_date"]

# Colonnes à supprimer si inutiles
DROP_COLUMNS = []          # ex: ["unnamed: 0"]

# Seuils
MISSING_THRESHOLD = 0.50   # supprimer si plus de 50% de valeurs manquantes
OUTLIER_IQR_FACTOR = 1.5

# Valeurs à considérer comme "manquantes"
MISSING_MARKERS = ["", " ", "NA", "N/A", "null", "None", "unknown", "Unknown"]

# Dictionnaire de standardisation des catégories
CATEGORY_MAPS = {
    # "gender": {"M": "Male", "F": "Female", "male": "Male", "female": "Female"},
    # "city": {"casa": "Casablanca", "CASA": "Casablanca"}
}

In [9]:
# Créer les dossiers si besoin
# Path("raw").mkdir(parents=True, exist_ok=True)
# Path("cleaned").mkdir(parents=True, exist_ok=True)

# Charger le dataset
df = pd.read_csv(RAW_PATH)

# Sauvegarde d'une copie de travail
df_raw = df.copy()

print("Dataset chargé avec succès.")
print("Shape:", df.shape)

Dataset chargé avec succès.
Shape: (10000, 8)


## 1. Compréhension du dataset

Questions à remplir :
- Une ligne représente quoi ?
- Quelle est la colonne ID ?
- Quelles colonnes sont critiques ?
- Quel est l’objectif final (dashboard / analyse / modèle / MVP) ?

In [ ]:
df.head()
df.sample(min(700en(df)))

print("Colonnes :", list(df.columns))
print("\nTypes :")
display(df.dtypes.to_frame("dtype"))

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
4373,TXN_6843677,NaN,ERROR,2.0,8.0,NaN,Takeaway,2023-09-09
7543,TXN_1048616,Tea,1,1.5,NaN,Cash,NaN,2023-10-10
274,TXN_4865338,Sandwich,UNKNOWN,4.0,16.0,NaN,In-store,2023-11-26
522,TXN_2947822,Cake,4,3.0,12.0,Digital Wallet,In-store,2023-10-27
9829,TXN_7300653,Cookie,5,1.0,5.0,ERROR,NaN,2023-04-24


Colonnes : ['Transaction ID', 'Item', 'Quantity', 'Price Per Unit', 'Total Spent', 'Payment Method', 'Location', 'Transaction Date']

Types :


,dtype
Transaction ID,str
Item,str
Quantity,str
Price Per Unit,str
Total Spent,str
Payment Method,str
Location,str
Transaction Date,str


In [ ]:
def profile_dataframe(dataframe):
    profile = pd.DataFrame({
        "column": dataframe.columns,
        "dtype": dataframe.dtypes.astype(str).values,
        "missing_count": dataframe.isna().sum().values,
        "missing_pct": (dataframe.isna().mean() * 100).round(2).values,
        "n_unique": dataframe.nunique(dropna=True).values
    })
    return profile.sort_values(by="missing_pct", ascending=False).reset_index(drop=True)

profile_before = profile_dataframe(df)
display(profile_before)

In [ ]:
# Uniformiser les valeurs manquantes
df = df.replace(MISSING_MARKERS, np.nan)

profile_after_missing_standardization = profile_dataframe(df)
display(profile_after_missing_standardization)

In [ ]:
print("Doublons exacts :", df.duplicated().sum())

if ID_COLUMNS:
    for col in ID_COLUMNS:
        if col in df.columns:
            dup_count = df.duplicated(subset=[col]).sum()
            print(f"Doublons sur {col} :", dup_count)

In [ ]:
for col in CATEGORICAL_COLUMNS:
    if col in df.columns:
        print(f"\n--- {col} ---")
        print(df[col].value_counts(dropna=False).head(20))

In [ ]:
if NUMERIC_COLUMNS:
    display(df[NUMERIC_COLUMNS].describe().T)

def detect_outliers_iqr(dataframe, col, factor=1.5):
    q1 = dataframe[col].quantile(0.25)
    q3 = dataframe[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - factor * iqr
    upper = q3 + factor * iqr
    return dataframe[(dataframe[col] < lower) | (dataframe[col] > upper)]

for col in NUMERIC_COLUMNS:
    if col in df.columns:
        outliers = detect_outliers_iqr(df.dropna(subset=[col]), col, OUTLIER_IQR_FACTOR)
        print(f"{col} -> outliers détectés : {len(outliers)}")

In [ ]:
# Convertir les dates
for col in DATE_COLUMNS:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

# Convertir les colonnes numériques
for col in NUMERIC_COLUMNS:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Nettoyage texte de base
for col in TEXT_COLUMNS + CATEGORICAL_COLUMNS:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype("string")
            .str.strip()
        )

In [ ]:
for col, mapping in CATEGORY_MAPS.items():
    if col in df.columns:
        df[col] = df[col].replace(mapping)

# Exemple optionnel : uniformiser la casse
for col in CATEGORICAL_COLUMNS:
    if col in df.columns:
        df[col] = df[col].str.strip().str.title()

In [ ]:
cols_to_drop = [col for col in DROP_COLUMNS if col in df.columns]
df = df.drop(columns=cols_to_drop, errors="ignore")

print("Colonnes supprimées :", cols_to_drop)
print("Shape actuelle :", df.shape)

In [ ]:
# 1) Supprimer les colonnes trop vides
too_missing = df.columns[df.isna().mean() > MISSING_THRESHOLD].tolist()
print("Colonnes > seuil manquants :", too_missing)

# Si tu veux les supprimer :
# df = df.drop(columns=too_missing)

# 2) Imputation simple
for col in NUMERIC_COLUMNS:
    if col in df.columns and df[col].isna().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

for col in CATEGORICAL_COLUMNS:
    if col in df.columns and df[col].isna().sum() > 0:
        mode_value = df[col].mode(dropna=True)
        fill_value = mode_value.iloc[0] if len(mode_value) > 0 else "Unknown"
        df[col] = df[col].fillna(fill_value)

for col in TEXT_COLUMNS:
    if col in df.columns:
        df[col] = df[col].fillna("Unknown")

In [ ]:
before_rows = len(df)

# Doublons exacts
df = df.drop_duplicates()

# Doublons par ID si applicable
for col in ID_COLUMNS:
    if col in df.columns:
        df = df.drop_duplicates(subset=[col], keep="first")

after_rows = len(df)

print("Lignes avant :", before_rows)
print("Lignes après :", after_rows)
print("Lignes supprimées :", before_rows - after_rows)

In [ ]:
validation_results = []

# 1. Colonnes obligatoires
for col in REQUIRED_COLUMNS:
    if col in df.columns:
        missing_count = df[col].isna().sum()
        validation_results.append({
            "rule": f"{col} doit être rempli",
            "status": "OK" if missing_count == 0 else "FAIL",
            "detail": f"missing_count={missing_count}"
        })

# 2. Unicité des IDs
for col in ID_COLUMNS:
    if col in df.columns:
        dup_count = df.duplicated(subset=[col]).sum()
        validation_results.append({
            "rule": f"{col} doit être unique",
            "status": "OK" if dup_count == 0 else "FAIL",
            "detail": f"duplicate_count={dup_count}"
        })

# 3. Valeurs négatives illogiques (exemple)
for col in NUMERIC_COLUMNS:
    if col in df.columns:
        negative_count = (df[col] < 0).sum()
        validation_results.append({
            "rule": f"{col} pas de valeur négative (à vérifier métier)",
            "status": "OK" if negative_count == 0 else "CHECK",
            "detail": f"negative_count={negative_count}"
        })

validation_df = pd.DataFrame(validation_results)
display(validation_df)

In [ ]:
summary_compare = pd.DataFrame({
    "metric": ["rows", "columns", "duplicate_rows", "total_missing_values"],
    "before": [
        df_raw.shape[0],
        df_raw.shape[1],
        df_raw.duplicated().sum(),
        df_raw.replace(MISSING_MARKERS, np.nan).isna().sum().sum()
    ],
    "after": [
        df.shape[0],
        df.shape[1],
        df.duplicated().sum(),
        df.isna().sum().sum()
    ]
})

display(summary_compare)

In [ ]:
df.to_csv(CLEAN_PATH, index=False)
print(f"Dataset nettoyé exporté vers : {CLEAN_PATH}")

In [ ]:
cleaning_log = pd.DataFrame([
    {"problem": "Valeurs manquantes", "action": "Remplacement / imputation", "justification": "Assurer la complétude minimale"},
    {"problem": "Doublons", "action": "Suppression des doublons", "justification": "Assurer l'unicité"},
    {"problem": "Formats incohérents", "action": "Standardisation", "justification": "Assurer la cohérence"},
    {"problem": "Types incorrects", "action": "Conversion types", "justification": "Permettre l'analyse correcte"},
])

display(cleaning_log)

# Optionnel : exporter la documentation
cleaning_log.to_csv("outputs_cleaning_log.csv", index=False)

## Conclusion

- Les données ont été profilées, nettoyées et validées.
- Les principales corrections ont porté sur :
  - valeurs manquantes
  - doublons
  - types / formats
  - standardisation
- Les limites restantes :
  - à compléter selon le dataset
- Le fichier nettoyé a été exporté dans `data/cleaned/`